# Computer Vision Pipeline Demo

This notebook is a lightweight companion to the realtime Tkinter application. It does **not** replace the desktop app. Real student data and biometric recognition models are intentionally not included in the public-ready repository.


## Pipeline

Camera / image → face detection → liveness → anti-spoofing → face recognition → roster validation → attendance.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print("Project root:", ROOT)
print("Python:", sys.version)


In [ ]:
# Dependency availability check
import importlib.util
for module in ["numpy", "cv2", "mediapipe", "tensorflow", "openpyxl"]:
    print(f"{module:12s}", "OK" if importlib.util.find_spec(module) else "MISSING")


## Model files

The public-ready repository intentionally excludes trained face-recognition model binaries and real class-index mappings. Copy your authorized local models into `models/` before running model inference.


In [ ]:
expected = [
    "anti_spoofing_mobilenet_real_spoof.h5",
    "face_recognition_mobilenet_pcd.h5",
    "class_indices_pcd.json",
]
for name in expected:
    p = ROOT / "models" / name
    print(("FOUND" if p.exists() else "MISSING"), p)


## MobileNet preprocessing example

The supplied application uses `tensorflow.keras.applications.mobilenet_v2.preprocess_input` for model input preprocessing.


In [ ]:
import numpy as np

def prepare_mobilenet_input(image_rgb, size=(224, 224)):
    import cv2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
    resized = cv2.resize(image_rgb, size)
    batch = np.expand_dims(resized.astype(np.float32), axis=0)
    return preprocess_input(batch)

print("Preprocessing helper defined.")


## Optional static-image anti-spoofing demo

Place an image you are authorized to use at `examples/sample_face.jpg`, and restore your local anti-spoofing model in `models/`. This cell will skip cleanly when those files are absent.


In [ ]:
sample = ROOT / "examples" / "sample_face.jpg"
model_path = ROOT / "models" / "anti_spoofing_mobilenet_real_spoof.h5"

if not sample.exists() or not model_path.exists():
    print("Demo skipped: provide examples/sample_face.jpg and the local anti-spoofing model.")
else:
    import cv2
    from tensorflow.keras.models import load_model
    bgr = cv2.imread(str(sample))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = prepare_mobilenet_input(rgb)
    model = load_model(model_path)
    pred = model.predict(x, verbose=0)
    print("Raw prediction:", pred)


## Why the realtime application stays in Python scripts

The main system uses a webcam loop, Tkinter GUI state, subprocess execution, session timers, and attendance report generation. Those concerns are easier to operate reliably as normal Python scripts. The notebook is best used for explanation, experiments, and reproducible static demonstrations.
